In [1]:
import pandas as pd
import numpy as np

from cleaning_pipeline import clean_agent_data

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Cleaning pipeline imported successfully!")

Pandas version: 3.0.3
NumPy version: 2.4.6
Cleaning pipeline imported successfully!


In [2]:
print("Jupyter kernel test successful")
print(10 + 20)

Jupyter kernel test successful
30


In [3]:
# Step 25: Load generated datasets

import pandas as pd
import numpy as np

from cleaning_pipeline import clean_agent_data

# Load the generated datasets
txns = pd.read_csv("data/agent_transactions_raw.csv")
agents = pd.read_csv("data/agents_raw.csv")

print("Raw transactions:", len(txns))
print("Registered agents:", len(agents))

# Run the cleaning pipeline
clean_df, log = clean_agent_data(txns, agents)

print("Cleaned transactions:", len(clean_df))

print("\nCleaning log:")
for entry in log:
    print("-", entry)

display(clean_df.head())

Raw transactions: 1015
Registered agents: 50
Cleaned transactions: 1000

Cleaning log:
- Removed 15 duplicate transaction IDs
- Standardised 493 transaction types
- Found 0 invalid transaction types
- Imputed 20 missing or invalid amounts
- Flagged 5 amount outliers for review
- Found 0 invalid transaction dates
- Merge preserved 1000 transactions; 10 have unknown agents


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,T00001,A011,Withdrawal,19875.0,2026-01-14,False,False,Laikipia
1,T00002,A016,Deposit,14690.0,2026-01-19,False,False,Nakuru
2,T00003,A042,Deposit,19669.0,2026-03-12,False,False,Kiambu
3,T00004,A049,Withdrawal,45075.0,2026-02-20,False,False,Laikipia
4,T00005,A049,Deposit,38638.0,2026-03-06,False,False,Laikipia


In [4]:
# Data Quality Audit Table

audit_df = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": int(txns["txn_id"].duplicated().sum())
    },
    {
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows affected": int(
            pd.to_numeric(txns["amount"], errors="coerce").isna().sum()
        )
    },
    {
        "issue": "Amount outliers",
        "dimension": "Accuracy",
        "column": "amount",
        "rows affected": int(clean_df["is_outlier"].sum())
    },
    {
        "issue": "Unknown agents",
        "dimension": "Consistency",
        "column": "agent_id",
        "rows affected": int(
            (~clean_df["agent_id"].isin(agents["agent_id"])).sum()
        )
    }
])

display(audit_df)

,issue,dimension,column,rows affected
0,Duplicate transaction IDs,Uniqueness,txn_id,15
1,Missing or invalid amounts,Completeness,amount,25
2,Amount outliers,Accuracy,amount,5
3,Unknown agents,Consistency,agent_id,10


In [5]:
# Step 15: Post-merge validation

rows_before_merge = len(
    txns.drop_duplicates(subset=["txn_id"])
)

rows_after_merge = len(clean_df)

unknown_agents = clean_df["county"].isna().sum()

validation_df = pd.DataFrame({
    "Validation check": [
        "Raw transactions",
        "Transactions after duplicate removal",
        "Transactions after cleaning and merge",
        "Transactions with unknown agents"
    ],
    "Count": [
        len(txns),
        rows_before_merge,
        rows_after_merge,
        unknown_agents
    ]
})

display(validation_df)

print("Rows before merge:", rows_before_merge)
print("Rows after merge:", rows_after_merge)
print("Unknown-agent transactions:", unknown_agents)

,Validation check,Count
0,Raw transactions,1015
1,Transactions after duplicate removal,1000
2,Transactions after cleaning and merge,1000
3,Transactions with unknown agents,10


Rows before merge: 1000
Rows after merge: 1000
Unknown-agent transactions: 10


In [6]:
# Step 16: County-level transaction summary

county_summary = (
    clean_df.groupby("county", dropna=False)
    .agg(
        total_transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean")
    )
    .reset_index()
    .sort_values("total_amount", ascending=False)
)

display(county_summary)

,county,total_transactions,total_amount,average_amount
2,Laikipia,219,7419812.0,33880.420091
4,Nairobi,205,5016341.5,24469.958537
3,Mombasa,173,4758707.5,27506.979769
1,Kisumu,153,4279438.0,27970.183007
5,Nakuru,162,4160356.0,25681.209877
0,Kiambu,78,2089217.0,26784.833333
6,NaN,10,243389.0,24338.9


In [7]:
# Step 17: Weekly transaction pivot table

# Convert transaction dates into datetime
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])

# Create a week column using the start of each week
clean_df["week"] = (
    clean_df["txn_date"]
    .dt.to_period("W")
    .dt.start_time
    .dt.strftime("%Y-%m-%d")
)

# Create the weekly pivot table
weekly_pivot = pd.pivot_table(
    clean_df,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    dropna=False
)

display(weekly_pivot)

week,2025-12-29,2026-01-05,2026-01-12,2026-01-19,2026-01-26,2026-02-02,2026-02-09,2026-02-16,2026-02-23,2026-03-02,2026-03-09,2026-03-16,2026-03-23,2026-03-30
county,,,,,,,,,,,,,,
Kiambu,159720.0,132681.0,117939.0,109905.0,151768.0,158755.0,50679.0,216784.5,163899.0,32180.0,187813.0,197847.5,384078.0,25168.0
Kisumu,151881.0,375673.0,321926.5,257290.0,427634.0,214947.0,450860.0,550165.5,222023.5,292606.0,391287.5,337840.0,185248.0,100056.0
Laikipia,111283.0,559342.5,773062.0,400647.0,612784.0,979924.0,326166.0,1036532.0,180192.0,608841.0,473383.5,473270.5,867238.5,17146.0
Mombasa,125393.0,285992.0,215584.5,321245.0,434586.0,591982.0,703235.5,293698.0,510082.5,494782.0,242409.0,177815.0,285516.0,76387.0
Nairobi,197878.0,256378.0,606131.0,450711.0,317292.0,422331.0,285512.0,412866.0,357539.0,295638.5,443968.5,406266.0,471507.5,92323.0
Nakuru,132376.0,252064.0,341627.0,316086.0,214753.0,317829.0,546787.0,350093.0,298250.0,303307.0,363203.5,384622.0,218330.0,121028.5
NaN,0.0,15673.0,0.0,0.0,90927.0,0.0,41113.0,29533.0,0.0,0.0,0.0,0.0,3880.0,62263.0


# Data Quality Report — Agent Transactions

## 1. Completeness
The raw-data audit identified 25 missing or invalid amount entries, including duplicate records. The cleaning pipeline imputes missing amounts using the median and records affected transactions using the `amount_imputed` flag.

## 2. Validity
Transaction types are standardised to `Deposit` and `Withdrawal`. Transaction amounts are converted to numeric values, and transaction dates are formatted as `YYYY-MM-DD`.

## 3. Accuracy
The pipeline identified 5 potential amount outliers using the interquartile range (IQR) method. These transactions were flagged using `is_outlier` and retained for further investigation.

## 4. Consistency
Transactions were merged with agent reference records using `agent_id`. The merge preserved 1,000 cleaned transactions, including 10 transactions with unknown agents.

## 5. Uniqueness
The generated dataset contained 1,015 transaction records. After removing 15 duplicate transaction IDs, 1,000 unique transactions remained.

## 6. Timeliness
Transaction dates were standardised to support chronological analysis and weekly reporting. However, the available data does not establish whether transactions were recorded promptly.

## Conclusion
The cleaning pipeline produced 1,000 cleaned transactions, a data-quality audit, a county summary, and a weekly pivot table.

**Limitation:** These results are based on a temporary seed-44 generator created for testing, not the original PLP dataset generator.

In [8]:
from pathlib import Path

# Create the output directory if needed
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Export the cleaned transactions
output_file = output_dir / "agent_transactions_clean.csv"

export_df = clean_df.copy()
export_df["txn_date"] = pd.to_datetime(
    export_df["txn_date"]
).dt.strftime("%Y-%m-%d")

export_df.to_csv(output_file, index=False)

print("CSV saved successfully:", output_file)
print("Rows exported:", len(export_df))
print("Columns:", export_df.columns.tolist())

CSV saved successfully: outputs\agent_transactions_clean.csv
Rows exported: 1000
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'week']


In [9]:
# Step 21: Validate exported CSV

validated_df = pd.read_csv(
    "outputs/agent_transactions_clean.csv"
)

print("Total rows:", len(validated_df))
print("Duplicate transaction IDs:",
      validated_df["txn_id"].duplicated().sum())

print("Invalid transaction types:",
      (~validated_df["txn_type"].isin(
          ["Deposit", "Withdrawal"]
      )).sum())

print("Missing amounts:",
      validated_df["amount"].isna().sum())

print("Missing counties:",
      validated_df["county"].isna().sum())

print("Amount numeric:",
      pd.api.types.is_numeric_dtype(validated_df["amount"]))

print("Required flag columns present:",
      {"amount_imputed", "is_outlier"}.issubset(
          validated_df.columns
      ))

print("Dates ISO formatted:",
      validated_df["txn_date"].str.fullmatch(
          r"\d{4}-\d{2}-\d{2}"
      ).all())

Total rows: 1000
Duplicate transaction IDs: 0
Invalid transaction types: 0
Missing amounts: 0
Missing counties: 10
Amount numeric: True
Required flag columns present: True
Dates ISO formatted: True
